In [3]:
import cv2
import numpy as np

def depth_estimation_from_yolo_and_midas(relative_depth_map, yolo_detections):
    """Estimates absolute depth from relative depth map and YOLO detections.

    Args:
        relative_depth_map: A colored image where brighter pixels are closer.
        yolo_detections: A list of YOLO detections, each with
            (x_min, y_min, x_max, y_max, depth).

    Returns:
        An absolute depth map (in meters).
    """

    # 1. Convert Relative Depth Map to Grayscale
    gray_depth_map = cv2.cvtColor(relative_depth_map, cv2.COLOR_BGR2GRAY)
    relative_depth = 1 - (gray_depth_map / 255.0)

    # 2. Estimate Scaling Factors from YOLO Bounding Boxes
    scaling_factors = []
    for detection in yolo_detections:
        x_min, y_min, x_max, y_max, depth = detection
        roi = relative_depth[y_min:y_max, x_min:x_max]
        mean_relative_depth = np.mean(roi)
        scaling_factors.append(depth / mean_relative_depth)

    # 3. Absolute Depth for Pixels Inside Bounding Boxes
    absolute_depth = np.zeros_like(relative_depth, dtype=np.float32)
    for i, detection in enumerate(yolo_detections):
        x_min, y_min, x_max, y_max, _ = detection
        absolute_depth[y_min:y_max, x_min:x_max] = (
            relative_depth[y_min:y_max, x_min:x_max] * scaling_factors[i]
        )

    # 4. Absolute Depth for Pixels Outside All Boxes
    height, width = relative_depth.shape
    for y in range(height):
        for x in range(width):
            if absolute_depth[y, x] == 0:  # Pixel outside boxes
                distances = []
                for detection in yolo_detections:
                    x_center = (detection[0] + detection[2]) // 2
                    y_center = (detection[1] + detection[3]) // 2
                    distance = np.sqrt((x - x_center)**2 + (y - y_center)**2)
                    distances.append(distance)

                # Inverse distance weights
                epsilon = 1e-6  # Small value to avoid division by zero
                weights = 1 / (np.array(distances) + epsilon)
                normalized_weights = weights / np.sum(weights)

                # Blend scaling factors
                scaling_factor = np.sum(normalized_weights * np.array(scaling_factors))

                # Compute absolute depth
                absolute_depth[y, x] = relative_depth[y, x] * scaling_factor

    return absolute_depth

In [ ]:

absolute_depth_map = depth_estimation_from_yolo_and_midas(relative_depth_map, yolo_detections)
